# Simulate Incremental Activity

**Purpose.** Keep the demo database lively without manual intervention. Each run
(the simulate-incremental notebook) inspects the watermark (`MAX(order_date)`) and fabricates enough new
business activity to fill the gap between then and `SYSUTCDATETIME()` -- so
reports stay current regardless of how often the pipeline runs.

This is the FIRST step in the iterative-load pipeline. Downstream notebooks
(silver, gold) run AFTER this and see fresh rows.

## What it does

1. Reads the order watermark from SQL.
2. Computes expected work: `(now - watermark) * TARGET_ORDERS_PER_DAY`.
3. If expected work crosses `SCALE_UP_THRESHOLD_ORDERS`, scales the DB to
   Gen5_8 via the Azure ARM REST API (granted to the workspace MSI), runs the
   load, then scales back to Gen5_4.
4. Generates, distributed uniformly over the gap window:
   - new customers (~1 per 200 orders)
   - orders + items + payments + shipments
   - returns (~5% of this run's items)
   - reviews (~30% of this run's orders)
   - inventory decrement (deducts `quantity_on_hand` based on items sold)
5. Exits with a JSON summary so a parent pipeline can log the activity.

## Parameters (top of next cell)

| Constant | Default | Meaning |
|---|---|---|
| `TARGET_ORDERS_PER_DAY` | 555 | Matches seed cadence (50K orders / 90 days) |
| `SCALE_UP_THRESHOLD_ORDERS` | 5000 | Above this expected count, scale up |
| `NEW_CUSTOMER_RATIO` | 0.005 | New customers per order (1 per 200) |
| `RETURN_RATIO` | 0.05 | Returns per item created this run |
| `REVIEW_RATIO` | 0.30 | Reviews per order created this run |


In [ ]:
# Parameters -- can be overridden by a pipeline that injects a parameter cell
TARGET_ORDERS_PER_DAY       = 555
SCALE_UP_THRESHOLD_ORDERS   = 5000
NEW_CUSTOMER_RATIO          = 0.005
RETURN_RATIO                = 0.05
REVIEW_RATIO                = 0.30

# Hard caps to keep one run from going off the rails (e.g. database hasn't
# been touched in months and the gap is enormous).
MAX_ORDERS_PER_RUN         = 200_000
MAX_GAP_DAYS                = 180

# These defaults are substituted at upload time by deploy.ps1 (same pattern as
# the seed notebook: empty-string defaults make the substitution unambiguous).
# For ad-hoc re-runs in the Fabric portal, edit these directly.
sql_server_fqdn   = ""
sql_database_name = "contoso_retail"
subscription_id   = ""
resource_group    = ""

print(f'SQL Server:  {sql_server_fqdn}')
print(f'SQL DB:      {sql_database_name}')
print(f'Subscription:{subscription_id}')
print(f'RG:          {resource_group}')


## Setup
Imports, AAD tokens, JDBC helpers (mirrors the seed notebook for self-containment).

In [ ]:
import datetime, random, uuid, json, time
import requests
from pyspark.sql import Row
from concurrent.futures import ThreadPoolExecutor, as_completed

random.seed()  # non-deterministic across runs

# --- stdlib stand-ins for the few "fake_*" helpers we need ----------------
_FIRST_NAMES = ["James","Mary","John","Patricia","Robert","Jennifer","Michael","Linda",
    "David","Elizabeth","William","Barbara","Richard","Susan","Joseph","Jessica",
    "Thomas","Sarah","Charles","Karen","Christopher","Nancy","Daniel","Lisa",
    "Matthew","Margaret","Anthony","Betty","Mark","Sandra","Donald","Ashley",
    "Steven","Kimberly","Paul","Emily","Andrew","Donna","Joshua","Michelle"]
_LAST_NAMES = ["Smith","Johnson","Williams","Brown","Jones","Garcia","Miller","Davis",
    "Rodriguez","Martinez","Hernandez","Lopez","Gonzalez","Wilson","Anderson","Thomas",
    "Taylor","Moore","Jackson","Martin","Lee","Perez","Thompson","White","Harris",
    "Sanchez","Clark","Ramirez","Lewis","Robinson","Walker","Young","Allen","King"]
_EMAIL_DOMAINS = ["gmail.com","yahoo.com","outlook.com","hotmail.com","icloud.com","aol.com"]
_STREETS = ["Main","Oak","Pine","Maple","Cedar","Elm","Washington","Lake","Hill",
    "Park","Walnut","Spring","Center","Mill","Chestnut","Highland","Forest","River"]
_STREET_SUFFIX = ["St","Ave","Rd","Blvd","Ln","Dr","Way","Ct","Pl"]
_CITIES = ["Springfield","Riverside","Franklin","Greenville","Bristol","Clinton",
    "Fairview","Salem","Madison","Georgetown","Arlington","Centerville","Burlington",
    "Manchester","Dover","Newport","Oxford","Auburn","Milton","Hudson","Kingston"]
_STATES = ["AL","AK","AZ","AR","CA","CO","CT","DE","FL","GA","HI","ID","IL","IN",
    "IA","KS","KY","LA","ME","MD","MA","MI","MN","MS","MO","MT","NE","NV","NH",
    "NJ","NM","NY","NC","ND","OH","OK","OR","PA","RI","SC","SD","TN","TX","UT",
    "VT","VA","WA","WV","WI","WY"]

def fake_first_name():  return random.choice(_FIRST_NAMES)
def fake_last_name():   return random.choice(_LAST_NAMES)
def fake_city():        return random.choice(_CITIES)
def fake_state_abbr():  return random.choice(_STATES)
def fake_zipcode():     return f"{random.randint(10000,99999)}"
def fake_street_address():
    return f"{random.randint(1,9999)} {random.choice(_STREETS)} {random.choice(_STREET_SUFFIX)}"
def fake_numerify(pattern):
    return "".join(str(random.randint(0,9)) if ch == "#" else ch for ch in pattern)
def fake_email(first, last):
    # This notebook uses time-jitter to keep emails unique across runs without a global dedupe set
    return f"{first.lower()}.{last.lower()}{random.randint(1000,99999)}@{random.choice(_EMAIL_DOMAINS)}"
def fake_date_of_birth(minimum_age=18, maximum_age=80):
    today = datetime.date.today()
    return today - datetime.timedelta(days=random.randint(minimum_age*365, maximum_age*365))


# AAD token for SQL. ARM token is fetched lazily inside scale_db() so a no-scale
# this notebook doesn't need ARM permissions.
sql_token = notebookutils.credentials.getToken('https://database.windows.net/')

jdbc_url = (
    f'jdbc:sqlserver://{sql_server_fqdn}:1433;'
    f'database={sql_database_name};'
    'encrypt=true;trustServerCertificate=false;'
    'hostNameInCertificate=*.database.windows.net;loginTimeout=30;'
)
jdbc_props = {
    'accessToken': sql_token,
    'driver': 'com.microsoft.sqlserver.jdbc.SQLServerDriver',
}

JDBC_BATCH_SIZE = 10_000
JDBC_NUM_PARTS  = 4   # run batches are smaller than seed, so 4 is plenty

def write_table(df, table_fqn, mode='append', num_parts=JDBC_NUM_PARTS):
    (df.repartition(num_parts)
       .write.format('jdbc')
       .option('url', jdbc_url)
       .option('dbtable', table_fqn)
       .option('batchsize', JDBC_BATCH_SIZE)
       .option('numPartitions', num_parts)
       .options(**jdbc_props)
       .mode(mode).save())
    print(f'  wrote -> {table_fqn}')

def write_tables_parallel(jobs, max_workers=4):
    def _run(j):
        if len(j) == 3:
            return write_table(j[0], j[1], num_parts=j[2])
        return write_table(j[0], j[1])
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        futs = {pool.submit(_run, j): j[1] for j in jobs}
        for f in as_completed(futs):
            f.result()
            print(f'  done: {futs[f]}')

def read_sql(query):
    return (spark.read.format('jdbc')
        .option('url', jdbc_url)
        .option('dbtable', f'({query}) t')
        .options(**jdbc_props)
        .load())

from pyspark.sql.types import (StructType, StructField,
    StringType, IntegerType, LongType, BooleanType, DateType, TimestampType,
    DecimalType)
from decimal import Decimal

# Explicit schemas matter when a a run produces a tiny batch -- Spark's
# inference on a single all-None nullable column yields NullType, which the
# JDBC writer refuses. These match the columns we INSERT to (identity PK and
# DEFAULT-only columns are omitted; SQL fills them in).
CUSTOMERS_SCHEMA = StructType([
    StructField('email',            StringType(),    False),
    StructField('first_name',       StringType(),    False),
    StructField('last_name',        StringType(),    False),
    StructField('phone',            StringType(),    True),
    StructField('date_of_birth',    DateType(),      True),
    StructField('segment_id',       IntegerType(),   False),
    StructField('address_line1',    StringType(),    True),
    StructField('address_line2',    StringType(),    True),
    StructField('city',             StringType(),    True),
    StructField('state',            StringType(),    True),
    StructField('postal_code',      StringType(),    True),
    StructField('country',          StringType(),    True),
    StructField('loyalty_tier',     StringType(),    False),
    StructField('loyalty_points',   IntegerType(),   False),
    StructField('marketing_opt_in', BooleanType(),   False),
    StructField('created_at',       TimestampType(), False),
    StructField('last_login_at',    TimestampType(), True),
    StructField('is_active',        BooleanType(),   False),
])
ORDERS_SCHEMA = StructType([
    StructField('order_number',       StringType(),       False),
    StructField('customer_id',        LongType(),         False),
    StructField('order_date',         TimestampType(),    False),
    StructField('order_status',       StringType(),       False),
    StructField('channel',            StringType(),       False),
    StructField('store_id',           IntegerType(),      True),
    StructField('subtotal',           DecimalType(12,2),  False),
    StructField('tax_amount',         DecimalType(10,2),  False),
    StructField('shipping_amount',    DecimalType(10,2),  False),
    StructField('discount_amount',    DecimalType(10,2),  False),
    StructField('total_amount',       DecimalType(12,2),  False),
    StructField('currency',           StringType(),       False),
    StructField('promotion_id',       IntegerType(),      True),
    StructField('ship_address_line1', StringType(),       True),
    StructField('ship_city',          StringType(),       True),
    StructField('ship_state',         StringType(),       True),
    StructField('ship_postal_code',   StringType(),       True),
    StructField('ship_country',       StringType(),       True),
])
ITEMS_SCHEMA = StructType([
    StructField('order_id',                 LongType(),        False),
    StructField('product_id',               LongType(),        False),
    StructField('quantity',                 IntegerType(),     False),
    StructField('unit_price',               DecimalType(10,2), False),
    StructField('line_discount',            DecimalType(10,2), False),
    StructField('line_total',               DecimalType(12,2), False),
    StructField('fulfillment_warehouse_id', IntegerType(),     True),
])
PAYMENTS_SCHEMA = StructType([
    StructField('order_id',        LongType(),        False),
    StructField('payment_method',  StringType(),      False),
    StructField('card_brand',      StringType(),      True),
    StructField('card_last_four',  StringType(),      True),
    StructField('amount',          DecimalType(12,2), False),
    StructField('status',          StringType(),      False),
    StructField('transaction_ref', StringType(),      True),
    StructField('processed_at',    TimestampType(),   False),
])
SHIPMENTS_SCHEMA = StructType([
    StructField('order_id',          LongType(),      False),
    StructField('warehouse_id',      IntegerType(),   False),
    StructField('carrier',           StringType(),    False),
    StructField('tracking_number',   StringType(),    True),
    StructField('shipped_at',        TimestampType(), True),
    StructField('estimated_delivery',DateType(),      True),
    StructField('delivered_at',      TimestampType(), True),
    StructField('status',            StringType(),    False),
])
RETURNS_SCHEMA = StructType([
    StructField('order_id',      LongType(),        False),
    StructField('order_item_id', LongType(),        False),
    StructField('customer_id',   LongType(),        False),
    StructField('return_reason', StringType(),      False),
    StructField('quantity',      IntegerType(),     False),
    StructField('refund_amount', DecimalType(10,2), False),
    StructField('return_status', StringType(),      False),
    StructField('requested_at',  TimestampType(),   False),
    StructField('completed_at',  TimestampType(),   True),
])
REVIEWS_SCHEMA = StructType([
    StructField('product_id',           LongType(),      False),
    StructField('customer_id',          LongType(),      False),
    StructField('order_id',             LongType(),      True),
    StructField('rating',               IntegerType(),   False),
    StructField('review_title',         StringType(),    True),
    StructField('review_text',          StringType(),    True),
    StructField('helpful_count',        IntegerType(),   False),
    StructField('created_at',           TimestampType(), False),
    StructField('is_verified_purchase', BooleanType(),   False),
])


## Discover watermark + compute batch size

In [ ]:
# Watermark: latest order date currently in SQL. New orders fill the gap from
# (watermark, now]. If table is empty (deploy without seed?), bail out.
wm_row = read_sql('SELECT MAX(order_date) AS wm, MIN(order_date) AS first_dt, COUNT(*) AS n FROM retail.orders').collect()[0]
watermark = wm_row['wm']
first_dt  = wm_row['first_dt']
total_n   = wm_row['n']

if watermark is None:
    raise RuntimeError('orders table is empty -- run the seed notebook first')

now = datetime.datetime.utcnow()
gap = now - watermark
gap_days = gap.total_seconds() / 86400.0

if gap_days <= 0:
    # Watermark is in the future (clock skew or back-dated seed); use a tiny window.
    gap_days = 0.01

# Clamp the gap so a long-idle env doesn't blow up
if gap_days > MAX_GAP_DAYS:
    print(f'  gap {gap_days:.1f}d clamped to {MAX_GAP_DAYS}d (use teardown/re-seed for full reset)')
    gap_days = MAX_GAP_DAYS
    watermark = now - datetime.timedelta(days=MAX_GAP_DAYS)

# Expected new orders, with a +/-15% jitter so runs aren't suspiciously identical.
base_orders = int(gap_days * TARGET_ORDERS_PER_DAY)
jitter = random.uniform(0.85, 1.15)
n_orders = max(1, min(MAX_ORDERS_PER_RUN, int(base_orders * jitter)))

window_start = watermark
window_end   = now
print(f'Watermark:      {watermark}')
print(f'Now (UTC):      {now}')
print(f'Gap:            {gap_days:.3f} days')
print(f'Expected orders:{n_orders:,} (target rate {TARGET_ORDERS_PER_DAY}/day x {gap_days:.2f}d, jitter {jitter:.2f})')
print(f'Existing orders:{total_n:,} (since {first_dt})')


## Conditional scale-up
Skip if the workload is tiny -- the scale operation itself (~30s up + ~30s down) costs more wall-clock than the writes would save.

In [ ]:
def scale_db(target_sku_name: str, target_capacity: int, target_min: float):
    # Fabric audience shortcuts vary by runtime; try common ones in order.
    arm_token = None
    for aud in ('https://management.azure.com/', 'https://management.core.windows.net/', 'arm', 'azure'):
        try:
            arm_token = notebookutils.credentials.getToken(aud)
            print(f'  got ARM token via audience: {aud!r}')
            break
        except Exception as e:
            print(f'  audience {aud!r} not supported: {e.__class__.__name__}')
    if not arm_token:
        raise RuntimeError('Could not obtain ARM token; scale-up unavailable')
    """PATCH the SQL DB SKU + min_capacity via ARM. Blocks until the operation
    completes (Azure SQL scale operations return immediately but resize takes
    ~30s; we poll the LRO until done so the seed phase doesn't fight a half-scaled DB)."""
    url = (f'https://management.azure.com/subscriptions/{subscription_id}'
           f'/resourceGroups/{resource_group}/providers/Microsoft.Sql/servers/'
           f'{sql_server_fqdn.split(".")[0]}/databases/{sql_database_name}'
           f'?api-version=2023-08-01-preview')
    body = {
        'sku': {'name': target_sku_name, 'tier': 'GeneralPurpose', 'family': 'Gen5', 'capacity': target_capacity},
        'properties': {'minCapacity': target_min},
    }
    headers = {'Authorization': f'Bearer {arm_token}', 'Content-Type': 'application/json'}
    r = requests.patch(url, headers=headers, data=json.dumps(body), timeout=60)
    if r.status_code not in (200, 201, 202):
        raise RuntimeError(f'Scale PATCH failed: {r.status_code} {r.text}')
    # 202 -> Async; poll Azure-AsyncOperation header until success
    if r.status_code == 202 and 'Azure-AsyncOperation' in r.headers:
        op = r.headers['Azure-AsyncOperation']
        deadline = time.time() + 600
        while time.time() < deadline:
            time.sleep(10)
            s = requests.get(op, headers={'Authorization': f'Bearer {arm_token}'}, timeout=30)
            status = s.json().get('status', 'unknown')
            print(f'  scale op status: {status}')
            if status in ('Succeeded',):
                return
            if status in ('Failed', 'Canceled'):
                raise RuntimeError(f'Scale op {status}: {s.text}')
        raise RuntimeError('Scale op timed out after 10 min')

scaled_up = False
if n_orders >= SCALE_UP_THRESHOLD_ORDERS:
    print(f'Scaling UP to GP_S_Gen5_8 (expected {n_orders:,} >= threshold {SCALE_UP_THRESHOLD_ORDERS:,})')
    scale_db('GP_S_Gen5_8', 8, 1.0)
    scaled_up = True
else:
    print(f'Skipping scale-up (expected {n_orders:,} < threshold {SCALE_UP_THRESHOLD_ORDERS:,})')


## Load reference data needed for generation

In [ ]:
# Just IDs + prices -- everything else stays in SQL.
products_back = read_sql('SELECT product_id, sku, list_price FROM retail.products WHERE is_active = 1').collect()
product_ids   = [int(r['product_id']) for r in products_back]
product_price = {int(r['product_id']): float(r['list_price']) for r in products_back}

customer_ids  = [int(r['customer_id']) for r in read_sql('SELECT customer_id FROM retail.customers WHERE is_active = 1').collect()]
segment_ids   = [int(r['segment_id']) for r in read_sql('SELECT segment_id FROM retail.customer_segments').collect()]
warehouse_ids = [int(r['warehouse_id']) for r in read_sql('SELECT warehouse_id FROM retail.warehouses').collect()]
store_ids     = [int(r['store_id']) for r in read_sql('SELECT store_id FROM retail.stores').collect()]
promo_ids     = [int(r['promotion_id']) for r in read_sql('SELECT promotion_id FROM retail.promotions').collect()]

print(f'  {len(product_ids):,} active products')
print(f'  {len(customer_ids):,} active customers')
print(f'  {len(warehouse_ids)} warehouses, {len(store_ids)} stores, {len(promo_ids)} promotions')


## Generate new customers (~1 per 200 orders)

In [ ]:
n_new_customers = max(0, int(n_orders * NEW_CUSTOMER_RATIO))
new_cust_rows = []
for _ in range(n_new_customers):
    created_at = window_start + datetime.timedelta(seconds=random.uniform(0, gap.total_seconds()))
    first = fake_first_name()
    last  = fake_last_name()
    new_cust_rows.append(Row(
        email=fake_email(first, last),
        first_name=first,
        last_name=last,
        phone=fake_numerify('+1##########'),
        date_of_birth=fake_date_of_birth(),
        segment_id=random.choice(segment_ids),
        address_line1=fake_street_address(),
        address_line2=None,
        city=fake_city(),
        state=fake_state_abbr(),
        postal_code=fake_zipcode(),
        country='USA',
        loyalty_tier=random.choices(['Bronze','Silver','Gold','Platinum'], weights=[60,25,12,3])[0],
        loyalty_points=random.randint(0, 500),
        marketing_opt_in=random.choice([True, False]),
        created_at=created_at,
        last_login_at=None,
        is_active=True,
    ))

if new_cust_rows:
    write_table(spark.createDataFrame(new_cust_rows, schema=CUSTOMERS_SCHEMA), 'retail.customers')
    # Refresh customer_ids so new orders can reference them
    customer_ids = [int(r['customer_id']) for r in read_sql('SELECT customer_id FROM retail.customers WHERE is_active = 1').collect()]
print(f'  wrote {len(new_cust_rows):,} new customers (total now {len(customer_ids):,})')


## Generate orders + items + payments + shipments

Order timestamps are distributed uniformly across the gap so runs at varying cadence still look natural in dashboards.

In [ ]:
channels         = ['online','store','mobile']
channel_weights  = [0.60, 0.20, 0.20]
payment_methods  = ['credit_card','debit_card','paypal','apple_pay','google_pay','store_credit','gift_card']
payment_weights  = [0.45,0.20,0.15,0.08,0.05,0.04,0.03]
card_brands      = ['Visa','Mastercard','Amex','Discover',None,None,None]
carriers         = ['UPS','FedEx','USPS','DHL']
# Simulated orders skew toward 'paid'/'shipped' (more recent -> less delivered yet)
order_statuses   = ['delivered','shipped','paid','cancelled']
status_weights   = [40, 35, 20, 5]

# Pick a per-run order_number offset that won't collide with seed numbers.
# Seed uses ORD-{i:010d} starting at 1; This notebook uses ORD-R-{run_uuid}-{i}.
run_id = uuid.uuid4().hex[:6].upper()  # 6 hex + 6 digits = 'ORD-R{6}-{6}' = 18 chars (NVARCHAR(20))
gap_seconds = (window_end - window_start).total_seconds()

orders_in = []
for i in range(n_orders):
    on = f'ORD-R{run_id}-{i+1:06d}'
    cust_id = random.choice(customer_ids)
    channel = random.choices(channels, weights=channel_weights)[0]
    store_id = random.choice(store_ids) if channel == 'store' else None
    promo_id = random.choice([None,None,None] + promo_ids)
    odate = window_start + datetime.timedelta(seconds=random.uniform(0, gap_seconds))
    n_items = random.choices([1,2,3,4,5], weights=[40,30,15,10,5])[0]

    items = []
    subtotal = 0.0; discount = 0.0
    for _ in range(n_items):
        pid = random.choice(product_ids)
        list_price = product_price[pid]
        qty = random.randint(1, 3)
        unit_price = round(list_price * random.uniform(0.85, 1.0), 2)
        line_gross = round(unit_price * qty, 2)
        line_disc  = round(line_gross * (0.1 if promo_id else 0), 2)
        line_total = round(line_gross - line_disc, 2)
        subtotal += line_gross; discount += line_disc
        items.append((pid, qty, unit_price, line_disc, line_total, random.choice(warehouse_ids)))

    tax = round((subtotal - discount) * 0.08, 2)
    shipping = 0.0 if (subtotal - discount) > 99 else round(random.uniform(5.99, 14.99), 2)
    total = round(subtotal - discount + tax + shipping, 2)
    status = random.choices(order_statuses, weights=status_weights)[0]

    orders_in.append({
        'order_number': on, 'customer_id': cust_id, 'order_date': odate,
        'order_status': status, 'channel': channel, 'store_id': store_id,
        'subtotal': round(subtotal,2), 'tax_amount': tax, 'shipping_amount': shipping,
        'discount_amount': round(discount,2), 'total_amount': total, 'currency': 'USD',
        'promotion_id': promo_id,
        'ship_address_line1': fake_street_address(), 'ship_city': fake_city(),
        'ship_state': fake_state_abbr(), 'ship_postal_code': fake_zipcode(), 'ship_country': 'USA',
        '_items': items,
    })

print(f'Prepared {len(orders_in):,} orders in driver memory')

# Insert orders, then resolve IDENTITY-assigned order_id values (same pattern as seed).
orders_only = [{k:v for k,v in o.items() if k != '_items'} for o in orders_in]
# Convert price/amount floats to Decimal so the DecimalType schema accepts them.
for o in orders_only:
    for k in ('subtotal','tax_amount','shipping_amount','discount_amount','total_amount'):
        o[k] = Decimal(str(o[k]))
orders_df = spark.createDataFrame([Row(**o) for o in orders_only], schema=ORDERS_SCHEMA)
write_table(orders_df, 'retail.orders')

# Use the run_id prefix to narrow the lookup -- way cheaper than pulling all orders.
order_id_map = {r['order_number']: int(r['order_id']) for r in
    read_sql(f"SELECT order_id, order_number FROM retail.orders WHERE order_number LIKE 'ORD-R{run_id}-%'").collect()}
print(f'Resolved {len(order_id_map):,} order_id values')

# Items / payments / shipments
item_rows = []; pay_rows = []; ship_rows = []
for o in orders_in:
    oid = order_id_map[o['order_number']]
    for (pid, qty, unit_price, line_disc, line_total, wh_id) in o['_items']:
        item_rows.append(Row(order_id=oid, product_id=pid, quantity=qty,
                             unit_price=unit_price, line_discount=line_disc,
                             line_total=line_total, fulfillment_warehouse_id=wh_id))
    pm = random.choices(payment_methods, weights=payment_weights)[0]
    cb = random.choice(card_brands) if 'card' in pm else None
    cl4 = fake_numerify('####') if cb else None
    pay_status = 'captured' if o['order_status'] != 'cancelled' else 'voided'
    pay_rows.append(Row(order_id=oid, payment_method=pm, card_brand=cb, card_last_four=cl4,
                        amount=o['total_amount'], status=pay_status,
                        transaction_ref=uuid.uuid4().hex.upper()[:20], processed_at=o['order_date']))
    if o['channel'] != 'store' and o['order_status'] not in ('cancelled','paid'):
        carrier = random.choice(carriers)
        tracking = f'{carrier[:3].upper()}{uuid.uuid4().hex[:12].upper()}'
        wh_id = random.choice(warehouse_ids)
        shipped = o['order_date'] + datetime.timedelta(hours=random.uniform(1, 36))
        if shipped > now: shipped = now
        est_del = shipped.date() + datetime.timedelta(days=random.randint(2,7))
        delivered = (shipped + datetime.timedelta(days=random.randint(1,5))) if o['order_status']=='delivered' else None
        if delivered and delivered > now: delivered = now
        ship_status = {'delivered':'delivered','shipped':'in_transit'}.get(o['order_status'], 'label_created')
        ship_rows.append(Row(order_id=oid, warehouse_id=wh_id, carrier=carrier, tracking_number=tracking,
                             shipped_at=shipped, estimated_delivery=est_del, delivered_at=delivered, status=ship_status))

print(f'Built {len(item_rows):,} items, {len(pay_rows):,} payments, {len(ship_rows):,} shipments')

# Items also have Decimal columns -- rebuild as tuples to match ITEMS_SCHEMA order.
items_tuples = [(r['order_id'], r['product_id'], r['quantity'],
                 Decimal(str(r['unit_price'])), Decimal(str(r['line_discount'])),
                 Decimal(str(r['line_total'])), r['fulfillment_warehouse_id'])
                for r in item_rows]
items_df    = spark.createDataFrame(items_tuples, schema=ITEMS_SCHEMA)
pay_tuples = [(r['order_id'], r['payment_method'], r['card_brand'], r['card_last_four'],
               Decimal(str(r['amount'])), r['status'], r['transaction_ref'], r['processed_at'])
              for r in pay_rows]
payments_df = spark.createDataFrame(pay_tuples, schema=PAYMENTS_SCHEMA)
ship_tuples = [(r['order_id'], r['warehouse_id'], r['carrier'], r['tracking_number'],
                r['shipped_at'], r['estimated_delivery'], r['delivered_at'], r['status'])
               for r in ship_rows]
shipments_df= spark.createDataFrame(ship_tuples, schema=SHIPMENTS_SCHEMA)
write_tables_parallel([
    (items_df,    'retail.order_items'),
    (payments_df, 'retail.payments'),
    (shipments_df,'retail.shipments'),
], max_workers=3)


## Returns + reviews for this run's orders

Returns target ~5% of new items, reviews target ~30% of new orders -- matches the
seed pattern so the cumulative ratios stay realistic.

In [ ]:
# Read back items with their resolved order_item_ids -- scoped to this run's orders.
run_order_ids_csv = ','.join(str(o) for o in order_id_map.values())
if not run_order_ids_csv:
    print('No Simulated orders -- skipping returns/reviews')
    items_back = []
else:
    # SQL has a 2100-parameter limit but order_id IN (a,b,c) with literals avoids that.
    # For very large run batches we chunk by 1000 to keep the IN list manageable.
    ids_list = list(order_id_map.values())
    items_back = []
    CHUNK = 1000
    for start in range(0, len(ids_list), CHUNK):
        chunk_csv = ','.join(str(x) for x in ids_list[start:start+CHUNK])
        items_back.extend(read_sql(
            f'SELECT order_item_id, order_id, product_id, quantity, line_total '
            f'FROM retail.order_items WHERE order_id IN ({chunk_csv})'
        ).collect())
print(f'Resolved {len(items_back):,} order_item_ids for this run')

order_meta = {order_id_map[o['order_number']]: (o['customer_id'], o['order_date']) for o in orders_in}

# RETURNS: ~5% of items
return_reasons = ['wrong_size','damaged','not_as_described','no_longer_needed',
                  'defective','wrong_item','quality_issue','arrived_late']
return_status_pool = ['refunded','refunded','refunded','received','requested','rejected']

ret_rows = []
for it in items_back:
    if random.random() >= RETURN_RATIO:
        continue
    cust_id, odate = order_meta[it['order_id']]
    ret_qty = random.randint(1, int(it['quantity']))
    refund = round(float(it['line_total']) * (ret_qty / int(it['quantity'])), 2)
    status = random.choice(return_status_pool)
    requested = odate + datetime.timedelta(hours=random.uniform(6, 72))
    if requested > now: requested = now
    completed = (requested + datetime.timedelta(hours=random.uniform(12, 96))) if status == 'refunded' else None
    if completed and completed > now: completed = now
    ret_rows.append(Row(
        order_id=int(it['order_id']),
        order_item_id=int(it['order_item_id']),
        customer_id=int(cust_id),
        return_reason=random.choice(return_reasons),
        quantity=int(ret_qty),
        refund_amount=float(refund),
        return_status=status,
        requested_at=requested,
        completed_at=completed,
    ))

# REVIEWS: ~30% of orders, one per order on a random item
items_by_order = {}
for it in items_back:
    items_by_order.setdefault(it['order_id'], []).append(it)

rating_weights = [3, 7, 15, 25, 50]
review_titles = {
    5: ['Excellent!','Love it','Highly recommend','Perfect','Worth every penny'],
    4: ['Pretty good','Solid choice','Mostly happy','Good value','Recommended'],
    3: ["It's okay",'Average','Met expectations','Decent','Not bad'],
    2: ['Disappointed','Below expectations',"Wouldn't buy again",'Had issues','Mediocre'],
    1: ['Terrible','Do not buy','Returned it','Worst purchase','Awful'],
}
review_text = {
    5: 'Exceeded my expectations -- showed up fast and quality is great.',
    4: "Good product overall. A couple minor nits but I'd buy again.",
    3: 'Works as advertised but nothing special. Fair for the price.',
    2: 'Build quality was below what I expected. Not great.',
    1: 'Defective on arrival. Wasted my time.',
}

rev_rows = []
for oid, its in items_by_order.items():
    if random.random() >= REVIEW_RATIO:
        continue
    pick = random.choice(its)
    cust_id, odate = order_meta[oid]
    rating = random.choices([1,2,3,4,5], weights=rating_weights)[0]
    created_at = odate + datetime.timedelta(hours=random.uniform(24, 240))
    if created_at > now: created_at = now
    rev_rows.append(Row(
        product_id=int(pick['product_id']),
        customer_id=int(cust_id),
        order_id=int(oid),
        rating=int(rating),
        review_title=random.choice(review_titles[rating]),
        review_text=review_text[rating],
        helpful_count=int(random.randint(0, 50)),
        created_at=created_at,
        is_verified_purchase=True,
    ))

print(f'Built {len(ret_rows):,} returns, {len(rev_rows):,} reviews')

writes = []
if ret_rows:
    ret_tuples = [(r['order_id'], r['order_item_id'], r['customer_id'],
                   r['return_reason'], r['quantity'], Decimal(str(r['refund_amount'])),
                   r['return_status'], r['requested_at'], r['completed_at'])
                  for r in ret_rows]
    writes.append((spark.createDataFrame(ret_tuples, schema=RETURNS_SCHEMA), 'retail.returns'))
if rev_rows:
    rev_tuples = [(r['product_id'], r['customer_id'], r['order_id'], r['rating'],
                   r['review_title'], r['review_text'], r['helpful_count'],
                   r['created_at'], r['is_verified_purchase'])
                  for r in rev_rows]
    writes.append((spark.createDataFrame(rev_tuples, schema=REVIEWS_SCHEMA), 'retail.reviews'))
if writes: write_tables_parallel(writes, max_workers=2)


## Inventory decrement

Sums quantities sold per `(product_id, warehouse_id)` for this run and decrements
`retail.inventory.quantity_on_hand` for the matching warehouse-location rows.

In [ ]:
# Aggregate decrements per (product_id, warehouse_id)
deltas = {}
for it in item_rows:
    key = (it['product_id'], it['fulfillment_warehouse_id'])
    deltas[key] = deltas.get(key, 0) + it['quantity']

if not deltas:
    print('No items sold -- skipping inventory decrement')
else:
    # Build delta DataFrame and stage to a temp table, then UPDATE...FROM in one statement.
    # Using a real (non-temp) table because JDBC writes go through their own session
    # and #temp tables don't survive across statements.
    stage_table = f'retail._sim_inv_delta_{run_id}'
    delta_rows = [Row(product_id=int(p), location_id=int(w), delta=int(q)) for (p,w),q in deltas.items()]
    delta_df = spark.createDataFrame(delta_rows)
    write_table(delta_df, stage_table, mode='overwrite', num_parts=1)

    # Run UPDATE...FROM via a raw JDBC connection through Spark's JVM gateway.
    # We can't use spark.write here because that's INSERT/OVERWRITE only -- the
    # JVM gateway exposes the same JDBC driver Spark already loaded.
    sc = spark.sparkContext
    jvm = sc._jvm
    DriverManager = jvm.java.sql.DriverManager
    props = jvm.java.util.Properties()
    props.setProperty('accessToken', sql_token)
    props.setProperty('driver', 'com.microsoft.sqlserver.jdbc.SQLServerDriver')
    conn = DriverManager.getConnection(jdbc_url, props)
    try:
        stmt = conn.createStatement()
        update_sql = f"""
            UPDATE inv
            SET quantity_on_hand = CASE WHEN inv.quantity_on_hand - d.delta < 0 THEN 0 ELSE inv.quantity_on_hand - d.delta END
            FROM retail.inventory inv
            JOIN {stage_table} d
              ON d.product_id = inv.product_id
             AND d.location_id = inv.location_id
             AND inv.location_type = 'warehouse'
        """
        affected = stmt.executeUpdate(update_sql)
        print(f'  decremented {affected:,} inventory rows')
        stmt.executeUpdate(f'DROP TABLE {stage_table}')
    finally:
        conn.close()


## Conditional scale-down + summary

In [ ]:
if scaled_up:
    print('Scaling back DOWN to GP_S_Gen5_4')
    scale_db('GP_S_Gen5_4', 4, 0.5)

summary = {
    'run_id': run_id,
    'window_start': window_start.isoformat(),
    'window_end': window_end.isoformat(),
    'gap_days': round(gap_days, 4),
    'scaled_up': scaled_up,
    'new_customers': len(new_cust_rows),
    'new_orders': len(orders_in),
    'new_items': len(item_rows),
    'new_payments': len(pay_rows),
    'new_shipments': len(ship_rows),
    'new_returns': len(ret_rows),
    'new_reviews': len(rev_rows),
    'inventory_locations_updated': len(deltas),
}
print(json.dumps(summary, indent=2))
notebookutils.notebook.exit(json.dumps(summary))
